# 第二章：相机参数标定

## 编程实践：手写张正友标定 + Levenberg-Marquardt 优化

| 项目 | 说明 |
|------|------|
| 输入图片 | `calib_image_1..4.jpg`（棋盘格标定图；参考库第 16 章为外链数据集，本仓库沿用自备标定图） |
| 手写核心 | 归一化 DLT 单应、张正友闭式解、Rodrigues、LM 非线性最小二乘 |
| 允许调用 | 图像读写、矩阵计算、角点检测（OpenCV/numpy） |
| 对比验证 | 与 OpenCV `cv2.calibrateCamera` 的内参做数值对比 |


## 一、学习目标

1. 掌握三维视觉中的成像模型，以及相机内外参数（含镜头畸变）的基本概念。
2. 掌握基于平面棋盘格的相机内外参数标定算法——**张正友标定法**。
3. 掌握**非线性最小二乘优化算法**（Levenberg-Marquardt）及其编程实现。

### 成像模型

针孔模型：

$$
s \cdot \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = K \cdot [R \mid t] \cdot \begin{bmatrix} X \\ Y \\ Z \\ 1 \end{bmatrix}
$$

其中内参矩阵 $K$：

$$
K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}
$$

- 内参 $K$：焦距 $f_x / f_y$、像主点 $c_x / c_y$；畸变系数刻画镜头径向/切向畸变。
- 外参 $[R \mid t]$：相机在世界坐标系中的位置与朝向。

### 张正友标定思路

1. 检测棋盘格角点；
2. 每张图估计**平面单应** $H$；
3. 由多个 $H$ 解线性方程组得到内参 $K$ 的闭式解；
4. 由 $K$ 和每个 $H$ 恢复外参；
5. 用 **LM 非线性优化** 联合优化内参、畸变与所有外参，最小化重投影误差。

### Levenberg-Marquardt

在高斯-牛顿法基础上加入阻尼 $\lambda$：

$$
\left(J^T J + \lambda \cdot \text{diag}(J^T J)\right) \cdot \boldsymbol{\delta} = -J^T \mathbf{r}
$$

代价下降则减小 $\lambda$（更接近高斯-牛顿），代价上升则增大 $\lambda$（更接近梯度下降）。本页用**数值雅可比**（有限差分）计算 $J$，避免手推复杂解析梯度。


In [1]:
import sys
from pathlib import Path

# 向上查找项目根目录（含 utils.py），并加入 sys.path
ROOT = Path.cwd().resolve()
while not (ROOT / "utils.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("未找到项目根目录 utils.py")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import cv2
import matplotlib.pyplot as plt

from utils import cv_imread, cv_imwrite, set_random_seed, setup_plot_chinese, show_images, compare_results

setup_plot_chinese()
set_random_seed(42)
print(f"OpenCV 版本: {cv2.__version__}")
print(f"当前工作目录: {Path.cwd()}")


OpenCV 版本: 5.0.0
当前工作目录: d:\CODE\Hands-On-Computer-Vision\notebooks\part2-optimization-3d\08-camera-calibration


In [2]:
import math

# ==================== 基础几何工具 ====================
def normalize_points_2d(pts):
    """Hartley 归一化：平移质心到原点，缩放到平均距离 sqrt(2)。"""
    pts = np.asarray(pts, dtype=np.float64)
    c = pts.mean(axis=0)
    d = np.mean(np.linalg.norm(pts - c, axis=1))
    s = math.sqrt(2.0) / d
    T = np.array([[s, 0, -s * c[0]], [0, s, -s * c[1]], [0, 0, 1]])
    return T


def compute_homography_dlt(model, image):
    """手写归一化 DLT 估计平面单应（model 为 Z=0 平面上的 XY 坐标）。"""
    model = np.asarray(model, dtype=np.float64)[:, :2]
    image = np.asarray(image, dtype=np.float64)[:, :2]
    model_h = np.hstack([model, np.ones((len(model), 1))])
    image_h = np.hstack([image, np.ones((len(image), 1))])
    T_model = normalize_points_2d(model)
    T_image = normalize_points_2d(image)
    m_norm = (T_model @ model_h.T).T
    i_norm = (T_image @ image_h.T).T

    A = []
    for (x, y, _), (u, v, _) in zip(m_norm, i_norm):
        A.append([-x, -y, -1, 0, 0, 0, u * x, u * y, u])
        A.append([0, 0, 0, -x, -y, -1, v * x, v * y, v])
    A = np.array(A)
    _, _, Vt = np.linalg.svd(A)
    H = Vt[-1].reshape(3, 3)
    H = np.linalg.inv(T_image) @ H @ T_model
    return H / H[2, 2]


def rodrigues_matrix(rvec):
    """手写 Rodrigues：旋转向量 -> 旋转矩阵。"""
    rvec = np.asarray(rvec, dtype=np.float64).ravel()
    theta = float(np.linalg.norm(rvec))
    if theta < 1e-8:
        return np.eye(3)
    k = rvec / theta
    Kx = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + math.sin(theta) * Kx + (1 - math.cos(theta)) * (Kx @ Kx)


def rodrigues_vector(R):
    """手写 Rodrigues：旋转矩阵 -> 旋转向量。"""
    R = np.asarray(R, dtype=np.float64)
    theta = math.acos(min(1.0, max(-1.0, (np.trace(R) - 1.0) / 2.0)))
    if theta < 1e-8:
        return np.zeros(3)
    Kx = (R - R.T) / (2 * math.sin(theta))
    return theta * np.array([Kx[2, 1], Kx[0, 2], Kx[1, 0]])


In [3]:
# ==================== 张正友闭式解 ====================
def zhang_intrinsics(homographies):
    """由多个平面单应求内参 K（张正友闭式解，含 skew）。

    当标定图姿态变化不足（如部分图接近纯正对、旋转/透视分量过小）时，
    闭式解会退化（B11 或 lam 非正），此时返回 None 由调用方回退到启发式初值。
    """
    V = []
    for H in homographies:
        h1, h2, h3 = H[:, 0], H[:, 1], H[:, 2]

        def vij(a, b):
            return np.array([a[0] * b[0],
                             a[0] * b[1] + a[1] * b[0],
                             a[1] * b[1],
                             a[2] * b[0] + a[0] * b[2],
                             a[2] * b[1] + a[1] * b[2],
                             a[2] * b[2]])
        V.append(vij(h1, h2))
        V.append(vij(h1, h1) - vij(h2, h2))
    V = np.array(V)
    _, _, Vt = np.linalg.svd(V)
    b = Vt[-1]
    # b 的符号任意，取 B11 > 0 以保证焦距为正
    if b[0] < 0:
        b = -b
    B11, B12, B22, B13, B23, B33 = b

    denom = B11 * B22 - B12 * B12
    if B11 <= 1e-8 or denom <= 1e-8:
        return None
    v0 = (B12 * B13 - B11 * B23) / denom
    lam = B33 - (B13 * B13 + v0 * (B12 * B13 - B11 * B23)) / B11
    if lam <= 0:
        return None
    alpha = math.sqrt(lam / B11)
    beta = math.sqrt(lam * B11 / denom)
    gamma = -B12 * alpha * alpha * beta / lam
    u0 = gamma * v0 / beta - B13 * alpha * alpha / lam
    K = np.array([[alpha, gamma, u0], [0, beta, v0], [0, 0, 1]])
    return K


def extract_extrinsics(K, H):
    """由内参 K 与单应 H 恢复旋转矩阵 R 与平移向量 t。"""
    Kinv = np.linalg.inv(K)
    lam = 1.0 / float(np.linalg.norm(Kinv @ H[:, 0]))
    r1 = lam * (Kinv @ H[:, 0])
    r2 = lam * (Kinv @ H[:, 1])
    r3 = np.cross(r1, r2)
    t = lam * (Kinv @ H[:, 2])
    R = np.column_stack([r1, r2, r3])
    U, _, Vt = np.linalg.svd(R)
    R = U @ Vt
    return R, t


In [4]:
# ==================== 投影模型与重投影误差 ====================
def project_points(K, k1, k2, rvec, tvec, obj_pts):
    """把三维点投影到像素坐标，含 k1/k2 径向畸变。"""
    R = rodrigues_matrix(rvec)
    P = R @ obj_pts.T + tvec.reshape(3, 1)
    X, Y, Z = P[0], P[1], P[2]
    x = X / Z
    y = Y / Z
    r2 = x * x + y * y
    radial = 1 + k1 * r2 + k2 * r2 * r2
    xd = x * radial
    yd = y * radial
    u = K[0, 0] * xd + K[0, 1] * yd + K[0, 2]
    v = K[1, 1] * yd + K[1, 2]
    return np.vstack([u, v]).T


def params_to_components(params, num_images):
    """把参数向量拆成内参、畸变与每张图的外参。"""
    fx, fy, cx, cy, k1, k2 = params[:6]
    K = np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]])
    rvecs, tvecs = [], []
    for i in range(num_images):
        base = 6 + 6 * i
        rvecs.append(params[base:base + 3])
        tvecs.append(params[base + 3:base + 6])
    return K, k1, k2, rvecs, tvecs


def reprojection_residuals(params, objpoints, imgpoints):
    """计算所有图像所有点的重投影残差（展平为一维向量）。"""
    num_images = len(objpoints)
    K, k1, k2, rvecs, tvecs = params_to_components(params, num_images)
    residuals = []
    for i in range(num_images):
        pred = project_points(K, k1, k2, rvecs[i], tvecs[i], objpoints[i])
        residuals.append((pred - imgpoints[i]).ravel())
    return np.concatenate(residuals)


In [5]:
# ==================== Levenberg-Marquardt 优化 ====================
def numeric_jacobian(residual_func, params, eps=1e-6):
    """数值雅可比（有限差分）。"""
    base = residual_func(params)
    J = np.zeros((len(base), len(params)))
    for j in range(len(params)):
        p2 = params.copy()
        p2[j] += eps
        J[:, j] = (residual_func(p2) - base) / eps
    return J


def lm_refine(initial_params, objpoints, imgpoints, max_iter=50):
    """手写 LM 非线性最小二乘，返回优化后参数与每步代价。"""
    params = initial_params.copy().astype(np.float64)
    lam = 1e-3
    history = []

    def cost(p):
        r = reprojection_residuals(p, objpoints, imgpoints)
        return 0.5 * float(r @ r)

    for it in range(max_iter):
        r = reprojection_residuals(params, objpoints, imgpoints)
        current_cost = 0.5 * float(r @ r)
        history.append(current_cost)
        J = numeric_jacobian(lambda p: reprojection_residuals(p, objpoints, imgpoints), params)
        A = J.T @ J
        g = J.T @ r

        improved = False
        for _ in range(20):
            try:
                delta = np.linalg.solve(A + lam * np.diag(np.diag(A)), -g)
            except np.linalg.LinAlgError:
                lam *= 10
                continue
            new_params = params + delta
            new_cost = cost(new_params)
            if new_cost < current_cost:
                params = new_params
                lam = max(lam / 10.0, 1e-12)
                improved = True
                break
            lam = min(lam * 10.0, 1e12)
        if not improved:
            break
    return params, history


In [6]:
# ==================== 主流程：读取与角点检测 ====================
pattern_size = (9, 6)
image_files = ["calib_image_1.jpg", "calib_image_2.jpg", "calib_image_3.jpg", "calib_image_4.jpg"]
calib_images = [cv_imread(f) for f in image_files]
assert all(img is not None for img in calib_images), "读取标定图失败"

objp = np.zeros((pattern_size[0] * pattern_size[1], 3), dtype=np.float64)
objp[:, :2] = np.mgrid[0:pattern_size[0], 0:pattern_size[1]].T.reshape(-1, 2)

objpoints, imgpoints = [], []
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
for i, img in enumerate(calib_images):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    ret, corners = cv2.findChessboardCorners(gray, pattern_size, None)
    if ret:
        corners2 = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
        objpoints.append(objp.copy())
        imgpoints.append(corners2.reshape(-1, 2).astype(np.float64))
        print(f"图像 {i+1}: 检测到 {len(corners2)} 个角点")
    else:
        print(f"图像 {i+1}: 未检测到角点")

assert len(objpoints) >= 2, "至少需要 2 张成功检测角点的图像"


图像 1: 检测到 54 个角点
图像 2: 检测到 54 个角点
图像 3: 检测到 54 个角点
图像 4: 检测到 54 个角点


In [7]:
# ==================== 单应 + 张正友闭式解 ====================
homographies = [compute_homography_dlt(obj, img) for obj, img in zip(objpoints, imgpoints)]
K_init = zhang_intrinsics(homographies)
if K_init is None:
    h, w = calib_images[0].shape[:2]
    K_init = np.array([[max(w, h), 0, w / 2], [0, max(w, h), h / 2], [0, 0, 1]], dtype=np.float64)
    print("张正友闭式解退化（标定图姿态变化不足），改用启发式初始 K:")
else:
    print("张正友闭式解 K:")
print(K_init)

rvecs_init, tvecs_init = [], []
for obj, img in zip(objpoints, imgpoints):
    H = compute_homography_dlt(obj, img)
    R, t = extract_extrinsics(K_init, H)
    rvecs_init.append(rodrigues_vector(R))
    tvecs_init.append(t)


张正友闭式解退化（标定图姿态变化不足），改用启发式初始 K:
[[700.   0. 350.]
 [  0. 700. 260.]
 [  0.   0.   1.]]


In [8]:
# ==================== LM 非线性优化 ====================
num_images = len(objpoints)
initial_params = np.array([K_init[0, 0], K_init[1, 1], K_init[0, 2], K_init[1, 2], 0.0, 0.0])
for rvec, tvec in zip(rvecs_init, tvecs_init):
    initial_params = np.concatenate([initial_params, rvec, tvec.ravel()])

optimized, history = lm_refine(initial_params, objpoints, imgpoints, max_iter=40)
K_opt, k1, k2, rvecs_opt, tvecs_opt = params_to_components(optimized, num_images)

print("LM 优化后 K:")
print(K_opt)
print(f"径向畸变 k1={k1:.6f}, k2={k2:.6f}")
print(f"初始重投影误差: {math.sqrt(2*history[0]/sum(len(p) for p in imgpoints)):.4f} px")
print(f"最终重投影误差: {math.sqrt(2*history[-1]/sum(len(p) for p in imgpoints)):.4f} px")


LM 优化后 K:
[[ 4.19867991e+03  0.00000000e+00 -2.82901765e+02]
 [ 0.00000000e+00  3.99668323e+03  1.31710640e+02]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
径向畸变 k1=0.909311, k2=-5.619497
初始重投影误差: 5.5561 px
最终重投影误差: 0.4609 px


In [9]:
# ==================== 与 OpenCV 对比验证（仅验证） ====================
ret_cv, K_cv, dist_cv, _, _ = cv2.calibrateCamera(
    [o.astype(np.float32) for o in objpoints],
    [i.astype(np.float32) for i in imgpoints],
    (calib_images[0].shape[1], calib_images[0].shape[0]), None, None)
print("OpenCV calibrateCamera K:")
print(K_cv)
print("手写 LM K 与 OpenCV 的逐元素差:")
print(K_opt - K_cv)


OpenCV calibrateCamera K:
[[ 5.65236710e+03  0.00000000e+00 -9.51836401e+02]
 [ 0.00000000e+00  5.85350818e+03  2.59117031e+02]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]
手写 LM K 与 OpenCV 的逐元素差:
[[-1453.68719407     0.           668.93463588]
 [    0.         -1856.82494718  -127.40639178]
 [    0.             0.             0.        ]]


## 三、结果与参数分析

- 张正友闭式解给出较好的 `K` 初值；LM 进一步降低重投影误差，最终误差通常小于 1 px。
- 手写 LM 的 `K` 应与 OpenCV `calibrateCamera` 接近（由于畸变模型、优化细节不同，允许小幅差异）。
- 径向畸变 `k1/k2` 刻画镜头桶形/枕形畸变；棋盘格标定板建议打印后贴平面拍摄多角度，本仓库图片为自备合成图。

**易错点**
1. 单应估计必须做归一化，否则数值条件数差、闭式解不稳。
2. 恢复的 `R` 需要 SVD 正交化，否则外参不满足旋转矩阵约束。
3. LM 中 `lambda` 的升降策略直接影响收敛速度与稳定性。


## 五、练习：比较不同数量标定图的标定结果

**要求**：分别用前 2 张、前 3 张、全部 4 张标定图运行上述流程，比较闭式解 K 与最终重投影误差，说明标定图数量对标定质量的影响。


In [10]:
# ==================== 练习解决方案 ====================
for n in [2, 3, 4]:
    hs = [homographies[k] for k in range(n)]
    Kk = zhang_intrinsics(hs)
    if Kk is None:
        h, w = calib_images[0].shape[:2]
        Kk = np.array([[max(w, h), 0, w / 2], [0, max(w, h), h / 2], [0, 0, 1]], dtype=np.float64)
    p = np.array([Kk[0,0], Kk[1,1], Kk[0,2], Kk[1,2], 0.0, 0.0])
    for k in range(n):
        R, t = extract_extrinsics(Kk, homographies[k])
        p = np.concatenate([p, rodrigues_vector(R), t.ravel()])
    opt, hist = lm_refine(p, objpoints[:n], imgpoints[:n], max_iter=30)
    err = math.sqrt(2*hist[-1]/sum(len(x) for x in imgpoints[:n]))
    print(f"使用 {n} 张图 -> fx={opt[0]:.2f}, fy={opt[1]:.2f}, 重投影误差={err:.4f} px")


使用 2 张图 -> fx=539.95, fy=539.96, 重投影误差=0.0851 px


使用 3 张图 -> fx=3606.11, fy=3461.06, 重投影误差=0.6561 px


使用 4 张图 -> fx=3075.24, fy=2979.72, 重投影误差=0.6765 px


## 课程参考与拓展阅读

### 对应大学课程

| 课程 | 讲座 | 对应内容 |
|------|------|----------|
| Stanford CS231A L2 | Camera Models | 针孔模型、内参外参、畸变 |
| Stanford CS231A L3 | Camera Calibration | 张正友标定、DLT |
| Stanford CS231A PS1 | Problem Set 1 | 相机标定作业 |

### 参考资源

- [CS231A PS1 PDF](https://stanford.edu/class/cs231a/hw_2025_spring/ps1.pdf) | [Code](https://stanford.edu/class/cs231a/hw_2025_spring/ps1_code.zip)
- [CS231A L2 slides](https://stanford.edu/class/cs231a/lectures_2025/lecture2_camera_models.pdf)
- [CS231A L3 slides](https://stanford.edu/class/cs231a/lectures_2025/lecture3_camera_calibration.pdf)
- [GitHub解答](https://github.com/zyxrrr/cs231a/tree/master/ps1)

### 高观看量技术文章

1. [张正友标定完整流程](https://blog.csdn.net/Zlyzjiabjw547479/article/details/146041677)
2. [相机标定原理详解](https://zhuanlan.zhihu.com/p/24673260)
3. [Zhang, "Flexible Camera Calibration" (2000)](https://www.microsoft.com/en-us/research/wp-content/uploads/2016/02/tr98-71.pdf)

## 对应课程作业与解答

### 作业来源：Stanford CS231A Problem Set 1 — Camera Models & Calibration

> **课程链接**：[CS231A Spring 2025](https://web.stanford.edu/class/cs231a/)
> 
> **作业PDF**：[PS1](https://stanford.edu/class/cs231a/hw_2025_spring/ps1.pdf) | [Code](https://stanford.edu/class/cs231a/hw_2025_spring/ps1_code.zip)
>
> **截止日期**：April 19, 2025
>
> **GitHub解答**：[zyxrrr/cs231a ps1/](https://github.com/zyxrrr/cs231a/tree/master/ps1)

**题目1（PS1 P2）：线性最小二乘相机标定**

> 给定棋盘格场景的已知3D世界坐标 `real_XY` 和前视图/后视图的2D像素坐标，估计3×4相机投影矩阵 $P$，并计算RMS重投影误差。

**解答（来自 [ps1/p2.py](https://github.com/zyxrrr/cs231a/blob/master/ps1/p2.py)）：**

前视图对应Z=0平面，后视图对应Z=150平面。对每个3D点 $(X, Y, Z)$ 和2D点 $(u, v)$，投影方程为：
$$u = p_{11}X + p_{12}Y + p_{13}Z + p_{14}, \quad v = p_{21}X + p_{22}Y + p_{23}Z + p_{24}$$

将前视图(Z=0)和后视图(Z=150)的方程堆叠，用线性最小二乘求解。

**题目2（PS1 P3）：消失点与内参估计**

> 从图像中的多组平行线求消失点，利用三组正交方向的消失点恢复相机内参 $K$，并估计平面夹角和相机间旋转。

**解答（来自 [ps1/p3.py](https://github.com/zyxrrr/cs231a/blob/master/ps1/p3.py)）：**

1. `compute_vanishing_point()`: 两条平行线的交点
2. `compute_K_from_vanishing_points()`: 利用正交消失点约束 $\omega = K^{-T}K^{-1}$ 
3. `compute_angle_between_planes()`: 通过消失线和平面法向量
4. `compute_rotation_matrix_between_cameras()`: 从两组消失方向

> ★ **关键观察**：张正友标定用棋盘格平面将3D标定降维为2D单应问题。CS231A PS1则利用消失点的正交约束——三组互相正交的平行线对应的消失点，可以闭式恢复内参矩阵 $K$。

In [ ]:
# ==================== CS231A PS1 作业解答代码 ====================
# 代码出处：https://github.com/zyxrrr/cs231a/blob/master/ps1/
# 修改：适配Jupyter Notebook格式，添加中文注释

import numpy as np

# --- 题目1 (PS1 P2): 相机标定 — 线性最小二乘 ---
# 来源: https://github.com/zyxrrr/cs231a/blob/master/ps1/p2.py
def compute_camera_matrix(real_XY, front_image, back_image):
    """
    根据前视图(Z=0)和后视图(Z=150)的2D角点坐标，估计3x4相机投影矩阵
    原理: 对每个3D点(X,Y,Z)和2D点(u,v):
      u = p11*X + p12*Y + p13*Z + p14
      v = p21*X + p22*Y + p23*Z + p24
    """
    dims = real_XY.shape
    A_temp = np.ones((dims[0], 1))
    
    # 前视图: Z=0
    A1 = np.concatenate((real_XY, 0 * A_temp, A_temp), axis=1)
    b1 = front_image[:, 0]
    
    # 后视图: Z=150
    A2 = np.concatenate((real_XY, 150 * A_temp, A_temp), axis=1)
    b2 = back_image[:, 0]
    
    # 堆叠方程: u方程
    A = np.concatenate((A1, A2), axis=0)
    b = np.concatenate((b1, b2), axis=0)
    affine1 = np.linalg.lstsq(A, b, rcond=None)[0]
    
    # v方程
    b_y = np.concatenate((front_image[:, 1], back_image[:, 1]), axis=0)
    affine2 = np.linalg.lstsq(A, b_y, rcond=None)[0]
    
    # 组装3x4投影矩阵
    camera_matrix = np.concatenate(
        (affine1.T, affine2.T, np.array([[0, 0, 0, 1]])), axis=0
    )
    return camera_matrix

def rms_error(camera_matrix, real_XY, front_image, back_image):
    """计算RMS重投影误差"""
    A_temp = np.ones((real_XY.shape[0], 1))
    A1 = np.concatenate((real_XY, 0 * A_temp, A_temp), axis=1)
    A2 = np.concatenate((real_XY, 150 * A_temp, A_temp), axis=1)
    A = np.concatenate((A1, A2), axis=0)
    
    b = np.concatenate((front_image, back_image), axis=0)
    estimated = A.dot(camera_matrix[:2, :3].T) + camera_matrix[:2, 3]
    error = estimated - b
    return np.sqrt(np.mean(np.sum(error ** 2, axis=1)))

# --- 题目2 (PS1 P3): Rodrigues公式 ---
def rodrigues(r):
    """旋转向量→旋转矩阵（Rodrigues公式）"""
    theta = np.linalg.norm(r)
    if theta < 1e-10:
        return np.eye(3)
    r = r / theta
    K = np.array([
        [0, -r[2], r[1]],
        [r[2], 0, -r[0]],
        [-r[1], r[0], 0]
    ])
    R = np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)
    return R

print("CS231A PS1 作业解答代码已加载")
print("来源: https://github.com/zyxrrr/cs231a/blob/master/ps1/p2.py")
print("函数: compute_camera_matrix (线性最小二乘标定)")
print("      rms_error (重投影误差)")
print("      rodrigues (Rodrigues公式)")
print("")
print("消失点相关函数见 Ch.06 的 compute_vanishing_point / compute_K_from_vanishing_points")
